In [ ]:
import pandas as pd
import os
import json
import tensorflow as tf
import numpy as np

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Import configuration file for paths
import sys
sys.path.insert(0, '/content/drive/MyDrive/CSCK507_EMA_GroupC_Final')
import config

# Load the cleaned data file
cleaned_data_file = os.path.join(config.MAIN_DIR_PATH, config.CLEANED_DATA_FILE)
df = pd.read_csv(cleaned_data_file)

Mounted at /content/drive


## **Text Vectorization**

In [ ]:
# Define special tokens
eos_token = "<EOS>"
sos_token = "<SOS>"
oov_token = "<OOV>"

# Convert the cleaned text to a list
vocab_list = df['cleaned_text'].tolist()

# Remove non-string values from vocab_list and convert any float to empty strings
vocab_list = [str(text) if isinstance(text, str) else '' for text in vocab_list]

# Initialize the tokenizer
tokenizer = tf.keras.preprocessing.text.Tokenizer(oov_token=oov_token)
tokenizer.fit_on_texts(vocab_list)

# Add <SOS> and <EOS> tokens to the tokenizer
tokenizer.word_index[sos_token] = len(tokenizer.word_index) + 1
tokenizer.word_index[eos_token] = len(tokenizer.word_index) + 1

# Convert the cleaned text into tokens
df['tokenized_text'] = tokenizer.texts_to_sequences(vocab_list)

# Reset the index of the DataFrame to ensure consistent alignment
df.reset_index(drop=True, inplace=True)

# Save the tokenizer as a JSON file for reuse
tokenizer_file = os.path.join(config.MAIN_DIR_PATH, config.TOKENIZER_FILE)
with open(tokenizer_file, 'w') as f:
    json.dump(tokenizer.to_json(), f)
print(f"Tokenizer saved to: {tokenizer_file}")

# Save the tokenized text as a new CSV file
tokenized_text_file = os.path.join(config.MAIN_DIR_PATH, config.TOKENIZED_DATA_FILE)
df.to_csv(tokenized_text_file, index=False)
print(f"Tokenized text saved to: {tokenized_text_file}")

Tokenizer saved to: /content/drive/MyDrive/CSCK507_EMA_GroupC_Final/tokenizer.json
Tokenized text saved to: /content/drive/MyDrive/CSCK507_EMA_GroupC_Final/tokenized_text.csv


## **Exploratory Data Analysis for Sequence Padding**

In [ ]:
# Explore data distribution to determine the maximum length of sequences for padding
# Calculate the length of each sequence
sequence_lengths = [len(seq) for seq in df['tokenized_text']]

# Calculate summary statistics
mean_length = np.mean(sequence_lengths)
median_length = np.median(sequence_lengths)
max_length = np.max(sequence_lengths)
percentile_75 = np.percentile(sequence_lengths, 75)
percentile_90 = np.percentile(sequence_lengths, 90)
percentile_95 = np.percentile(sequence_lengths, 95)
percentile_99 = np.percentile(sequence_lengths, 99)
print(f"Mean sequence length: {mean_length}")
print(f"Median sequence length: {median_length}")
print(f"Max sequence length: {max_length}")
print(f"75th percentile of sequence lengths: {percentile_75}")
print(f"90th percentile of sequence lengths: {percentile_90}")
print(f"95th percentile of sequence lengths: {percentile_95}")
print(f"99th percentile of sequence lengths: {percentile_99}")

Mean sequence length: 10.382371114192456
Median sequence length: 8.0
Max sequence length: 198
75th percentile of sequence lengths: 14.0
90th percentile of sequence lengths: 22.0
95th percentile of sequence lengths: 28.0
99th percentile of sequence lengths: 46.0


## **Input-Output Pairs Creation & Sequence Padding**

In [ ]:
# Function to pad sequences with a maximum length of 30 and truncate longer sequences
def pad_sequences(sequence):
    return tf.keras.preprocessing.sequence.pad_sequences(
        sequence,
        maxlen=30,         # Limit sequences to 30 tokens
        padding='post',    # Pad at the end of the sequence
        truncating='post'  # Truncate sequences longer than 30 tokens at the end
    )

In [ ]:
def generate_pairs(df, tokenized_sequences_array, sos_token_idx, eos_token_idx, batch_size=1000):
    encoder_input_seq= []
    decoder_input_seq = []
    target_sequences = []

    # Iterate over grouped sequences by both 'folder' and 'dialogueID'
    for (folder, dialogue_id), group in df.groupby(['folder', 'dialogueID']):
        indices = group.index.tolist()
        dialogue_turns = tokenized_sequences_array[indices]

        # If the dialogue has an odd number of turns, ignore the last one
        max_turn_idx = len(dialogue_turns) - 1
        if len(dialogue_turns) % 2 != 0:
            max_turn_idx -= 1  # Ignore the last turn

        # Process consecutive turns: input is dialogue_turns[i], target is dialogue_turns[i + 1]
        for i in range(max_turn_idx):
            # Current turn (input) for the encoder
            encoder_input_seq.append(dialogue_turns[i])

            # Next turn (target) for the decoder:
            # 1. Add <SOS> to the beginning of the decoder input
            decoder_input_seq.append(np.concatenate([[sos_token_idx], dialogue_turns[i + 1]]))

            # 2. Add <EOS> to the end of the target sequence
            target_sequences.append(np.concatenate([dialogue_turns[i + 1], [eos_token_idx]]))

            # Yield data in batches
            if len(encoder_input_seq) >= batch_size:
                # Apply the padding_sequence function to all the sequences
                padded_encoder_input_seq = pad_sequences(encoder_input_seq)
                padded_decoder_input_seq = pad_sequences(decoder_input_seq)
                padded_target_sequences = pad_sequences(target_sequences)

                yield np.array(padded_encoder_input_seq), np.array(padded_decoder_input_seq), np.array(padded_target_sequences)

                # Reset batch sequences
                encoder_input_seq= []
                decoder_input_seq = []
                target_sequences = []

    # Yield any remaining sequences that didn't fill a full batch
    if encoder_input_seq:
        padded_encoder_input_seq = pad_sequences(encoder_input_seq)
        padded_decoder_input_seq = pad_sequences(decoder_input_seq)
        padded_target_sequences = pad_sequences(target_sequences)

        yield np.array(padded_encoder_input_seq), np.array(padded_decoder_input_seq), np.array(padded_target_sequences)

In [ ]:
#  Get the indices of <sos> and <eos> tokens
sos_token_idx = tokenizer.word_index[sos_token]
eos_token_idx = tokenizer.word_index[eos_token]

# Convert the 'tokenized_text' column to a NumPy array for better performance
tokenized_sequences_array = np.array(df['tokenized_text'].tolist(), dtype=object)

# Collect all the data into a list
encoder_input = []
decoder_input = []
target = []

# Collect all batches into lists
for X_enc_batch, X_dec_batch, y_batch in generate_pairs(df, tokenized_sequences_array, sos_token_idx, eos_token_idx, batch_size=1000):
    encoder_input.append(X_enc_batch)
    decoder_input.append(X_dec_batch)
    target.append(y_batch)

# Convert the lists into NumPy arrays (concatenating all batches)
X_enc = np.concatenate(encoder_input)
X_dec = np.concatenate(decoder_input)
y = np.concatenate(target)

# Save the final input-output padded pairs
input_output_file = os.path.join(config.MAIN_DIR_PATH, config.INPUT_OUTPUT_PAIRS)
np.savez_compressed(input_output_file, X_enc=X_enc, X_dec=X_dec, y=y)
print(f"Input-output padded pairs saved to: {input_output_file}")

Input-output padded pairs saved to: /content/drive/MyDrive/CSCK507_EMA_GroupC_Final/input_output_pairs.npz


## **Data Split into Training, Validation and Test Datasets**

In [ ]:
from sklearn.model_selection import train_test_split

# First split: Train (70%) and Temp (30%)
X_train_enc, X_temp_enc, X_train_dec, X_temp_dec, y_train, y_temp = train_test_split(
    X_enc, X_dec, y, test_size=0.3, random_state=42)

# Second split: Temp (30%) split into Validation (15%) and Test (15%)
X_val_enc, X_test_enc, X_val_dec, X_test_dec, y_val, y_test = train_test_split(
    X_temp_enc, X_temp_dec, y_temp, test_size=0.5, random_state=42)

# Output shapes for verification
print(f"Train encoder shape: {X_train_enc.shape}, Train decoder shape: {X_train_dec.shape}, Train target shape: {y_train.shape}")
print(f"Validation encoder shape: {X_val_enc.shape}, Validation decoder shape: {X_val_dec.shape}, Validation target shape: {y_val.shape}")
print(f"Test encoder shape: {X_test_enc.shape}, Test decoder shape: {X_test_dec.shape}, Test target shape: {y_test.shape}")

# Save the train, validation, and test sets separately
np.savez_compressed(os.path.join(config.MAIN_DIR_PATH, config.TRAIN_DATA), X_train_enc=X_train_enc, X_train_dec=X_train_dec, y_train=y_train)
np.savez_compressed(os.path.join(config.MAIN_DIR_PATH, config.VAL_DATA), X_val_enc=X_val_enc, X_val_dec=X_val_dec, y_val=y_val)
np.savez_compressed(os.path.join(config.MAIN_DIR_PATH, config.TEST_DATA), X_test_enc=X_test_enc, X_test_dec=X_test_dec, y_test=y_test)
print("Training, Validation, and Test sets saved.")

Train encoder shape: (9451799, 30), Train decoder shape: (9451799, 30), Train target shape: (9451799, 30)
Validation encoder shape: (2025386, 30), Validation decoder shape: (2025386, 30), Validation target shape: (2025386, 30)
Test encoder shape: (2025386, 30), Test decoder shape: (2025386, 30), Test target shape: (2025386, 30)
Training, Validation, and Test sets saved.
